In [1]:
import subprocess, sys, os, json, pickle
import warnings
warnings.filterwarnings("ignore")

IN_COLAB = False
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    pass

for pkg in ["lightgbm", "catboost", "xgboost", "shap", "pyarrow"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except subprocess.CalledProcessError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", pkg])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titleweight"] = "bold"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

TIER_ORDER = ["Low", "Watch", "Warning", "Alert"]
TIER_TO_INT = {t: i for i, t in enumerate(TIER_ORDER)}
INT_TO_TIER = {i: t for t, i in TIER_TO_INT.items()}

def tier_mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.linear_model import LogisticRegression

print(f"Setup complete. Colab: {IN_COLAB}, Python: {sys.version.split()[0]}")


Installing catboost...
Setup complete. Colab: True, Python: 3.12.13


## Upload the raw dataset



In [2]:
if IN_COLAB:
    print("Choose dengueradar_training_table.csv from your computer...")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
else:

    candidates = [
        "dengueradar_training_table.csv",
        "../dengueradar_training_table.csv",
        "/workspace/dengueradar_training_table.csv",
        "/content/dengueradar_training_table.csv",
    ]
    DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
    if DATA_PATH is None:
        raise FileNotFoundError(
            "Place dengueradar_training_table.csv in the current directory or update DATA_PATH."
        )
    print(f"Found data at: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print(f"Date range: {pd.to_datetime(df_raw['week_start']).min().date()} -> {pd.to_datetime(df_raw['week_start']).max().date()}")
print(f"MOH areas: {df_raw['moh_name'].nunique()}  |  Districts: {df_raw['district'].nunique()}")


Choose dengueradar_training_table.csv from your computer...


Saving dengueradar_training_table.csv to dengueradar_training_table.csv
Loaded: 145,985 rows, 28 columns
Date range: 2013-12-28 -> 2026-05-11
MOH areas: 226  |  Districts: 25


##  data overview



In [3]:
df = df_raw.copy()
df["week_start"] = pd.to_datetime(df["week_start"])
df = df.sort_values(["moh_name", "week_start"]).reset_index(drop=True)

for col in ["cases_lag1", "cases_lag2"]:
    df[col] = df.groupby("moh_name")[col].transform(lambda s: s.bfill())

print("Risk tier distribution (original):")
print(df["risk_tier"].value_counts().reindex(TIER_ORDER))
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = df["risk_tier"].value_counts().reindex(TIER_ORDER)
axes[0].bar(counts.index, counts.values, color=["#3a7d44","#f9a03f","#e63946","#9d0208"])
axes[0].set_title("Risk tier counts"); axes[0].set_ylabel("count")
for i, v in enumerate(counts.values): axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")
pct = counts / counts.sum() * 100
axes[1].pie(pct, labels=[f"{t}\n{p:.1f}%" for t, p in zip(TIER_ORDER, pct)],
            colors=["#3a7d44","#f9a03f","#e63946","#9d0208"], startangle=90)
axes[1].set_title("Risk tier proportion")
plt.tight_layout(); plt.show()

print("Class imbalance (Low:Alert):", round(counts["Low"] / counts["Alert"], 1), ": 1")
print("This is why we need class weights later - the model would just predict Low for everything otherwise.")


Risk tier distribution (original):
risk_tier
Low        73743
Watch      30818
Warning    22875
Alert      18549
Name: count, dtype: int64

Class imbalance (Low:Alert): 4.0 : 1
This is why we need class weights later - the model would just predict Low for everything otherwise.


##  Preprocessing

 The original `risk_tier` column was defined using percentile thresholds computed on the *entire* dataset. That means test period labels were partially defined by information from the test period itself - a subtle form of target leakage that inflated v1's accuracy.

In [4]:
train_mask = df["week_start"] < "2024-01-01"
val_mask   = (df["week_start"] >= "2024-01-01") & (df["week_start"] < "2025-01-01")
test_mask  = df["week_start"] >= "2025-01-01"

thresholds = df.loc[train_mask, "incidence_per_100k"].quantile([0.50, 0.80, 0.95]).values
T1, T2, T3 = thresholds
print(f"Tier thresholds (from training data 2014-2023):")
print(f"  Low:      incidence < {T1:.2f}")
print(f"  Watch:    {T1:.2f} <= incidence < {T2:.2f}")
print(f"  Warning:  {T2:.2f} <= incidence < {T3:.2f}")
print(f"  Alert:    incidence >= {T3:.2f}")

def to_tier(inc):
    if inc < T1: return 0
    if inc < T2: return 1
    if inc < T3: return 2
    return 3
df["risk_tier_int"] = df["incidence_per_100k"].apply(to_tier).astype(int)

train = df[train_mask].copy()
val   = df[val_mask].copy()
test  = df[test_mask].copy()
for name, part in [("Train 2014-2023", train), ("Val 2024", val), ("Test 2025-2026", test)]:
    mix = pd.Series(part["risk_tier_int"]).map(INT_TO_TIER).value_counts().reindex(TIER_ORDER).to_dict()
    print(f"  {name}: {len(part):>7,} rows, tier mix = {mix}")


Tier thresholds (from training data 2014-2023):
  Low:      incidence < 2.75
  Watch:    2.75 <= incidence < 7.94
  Alert:    incidence >= 23.03
  Train 2014-2023: 118,198 rows, tier mix = {'Low': 59078, 'Watch': 35479, 'Warning': 17731, 'Alert': 5910}
  Val 2024:  11,752 rows, tier mix = {'Low': 4677, 'Watch': 4935, 'Warning': 1923, 'Alert': 217}
  Test 2025-2026:  16,035 rows, tier mix = {'Low': 5327, 'Watch': 6877, 'Warning': 3393, 'Alert': 438}


## Feature engineering (63 features)

We build 63 features per (MOH, week) row. The model predicts the tier for week **t+1** using features computed only from data **<= t** - no future information.

**Feature groups (all lagged, no leakage):**
1. Case lags at 1, 2, 3, 4, 5, 8, 12, 26, 52 weeks (year-over-year!)
2. Rolling statistics (mean/max/std over 4/8/12 weeks)
3. Growth rate & acceleration
4. Long-term linear trend
5. Incidence (cases per 100k) lags
6. Seasonality (month + week-of-year as sin/cos)
7. District-level aggregates (lagged)
8. Spatial rank within district (using last week's cases)
9. Weather + interactions
10. Population (log-transformed)
11. Time since last outbreak (lagged)

In [5]:
g = df.groupby("moh_name")["cases"]

for lag in [1, 2, 3, 4, 5, 8, 12, 26, 52]:
    df[f"cases_lag{lag}"] = g.shift(lag)

_shifted = df.groupby("moh_name")["cases"].shift(1)
for window in [4, 8, 12]:
    df[f"cases_roll{window}_mean"] = _shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    df[f"cases_roll{window}_max"]  = _shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).max().reset_index(level=0, drop=True)
    df[f"cases_roll{window}_std"]  = _shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).std().reset_index(level=0, drop=True)

df["case_growth_wow"] = ((g.shift(1).values - g.shift(2).values) / (g.shift(2).values + 1.0)).clip(-5, 5)
df["case_accel"] = df.groupby("moh_name")["case_growth_wow"].diff()
def _rolling_slope(s, window=8):
    def slope(arr):
        if len(arr) < 2 or np.std(arr) < 1e-6: return 0.0
        return float(np.polyfit(np.arange(len(arr)), arr, 1)[0])
    return s.rolling(window, min_periods=2).apply(slope, raw=True)
df["case_trend_8w"] = _shifted.groupby(df["moh_name"]).transform(lambda s: _rolling_slope(s, 8))

for lag in [1, 2, 4, 8]:
    df[f"inc_lag{lag}"] = df.groupby("moh_name")["incidence_per_100k"].shift(lag)
inc_shifted = df.groupby("moh_name")["incidence_per_100k"].shift(1)
for window in [4, 12]:
    df[f"inc_roll{window}_mean"] = inc_shifted.groupby(df["moh_name"]).rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)

df["month"] = df["week_start"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["woy"] = df["week_start"].dt.isocalendar().week.astype(int)
df["woy_sin"] = np.sin(2 * np.pi * df["woy"] / 52)
df["woy_cos"] = np.cos(2 * np.pi * df["woy"] / 52)
df["iso_year"] = df["week_start"].dt.isocalendar().year.astype(int)

district_week = (df.groupby(["district","week_start"])
                 .agg(district_total=("cases","sum"), district_mean=("cases","mean"), district_max=("cases","max"))
                 .reset_index().sort_values(["district","week_start"]))
for lag in [1, 2, 4]:
    district_week[f"district_total_lag{lag}"] = district_week.groupby("district")["district_total"].shift(lag)
    district_week[f"district_mean_lag{lag}"]  = district_week.groupby("district")["district_mean"].shift(lag)
    district_week[f"district_max_lag{lag}"]   = district_week.groupby("district")["district_max"].shift(lag)
shifted_total = district_week.groupby("district")["district_total"].shift(1)
district_week["district_total_roll4"]  = shifted_total.groupby(district_week["district"]).rolling(4, min_periods=1).mean().reset_index(level=0, drop=True)
district_week["district_total_roll12"] = shifted_total.groupby(district_week["district"]).rolling(12, min_periods=1).mean().reset_index(level=0, drop=True)
df = df.merge(
    district_week[["district","week_start",
                   "district_total_lag1","district_total_lag2","district_total_lag4",
                   "district_mean_lag1","district_max_lag1",
                   "district_total_roll4","district_total_roll12"]],
    on=["district","week_start"], how="left")

prev_cases = df.groupby("moh_name")["cases"].shift(1)
df["_p"] = prev_cases
df["district_rank_lag1"] = df.groupby(["district","week_start"])["_p"].rank(pct=True)
df["district_zscore_lag1"] = ((df["_p"] - df.groupby(["district","week_start"])["_p"].transform("mean")) /
                              (df.groupby(["district","week_start"])["_p"].transform("std") + 1e-3))
df = df.drop(columns=["_p"])

df["temp_range"]   = df["temp_max"] - df["temp_min"]
df["rain_change"]  = df["rain_1w"] - df["rain_2w"] / 2.0
df["heat_index"]   = df["temp_avg"] * df["humidity"] / 100.0
df["rain_x_temp"]  = df["rain_1w"] * df["temp_avg"]
df["rain_x_humid"] = df["rain_1w"] * df["humidity"]
df["log_pop"]      = np.log1p(df["population"])
df["log_density"]  = np.log1p(df["pop_density"])

def _wst(s, threshold=20):
    c, out = 999, []
    for v in s:
        if v > threshold: c = 0
        else: c += 1
        out.append(min(c, 52))
    return pd.Series(out, index=s.index)
df["_lc"] = df.groupby("moh_name")["cases"].shift(1)
df["weeks_since_outbreak_lag1"] = df.groupby("moh_name")["_lc"].transform(_wst)
df = df.drop(columns=["_lc"])

all_districts = sorted(df["district"].unique())
district_to_idx = {d: i for i, d in enumerate(all_districts)}
df["district_cat"] = df["district"].map(district_to_idx).astype("int32")

lag_cols = [c for c in df.columns if "lag" in c or "_roll" in c or c in ["case_trend_8w","case_growth_wow","case_accel"]]
df[lag_cols] = df.groupby("moh_name")[lag_cols].transform(lambda s: s.ffill().bfill()).fillna(0)
df["weeks_since_outbreak_lag1"] = df["weeks_since_outbreak_lag1"].fillna(52)

DROP_COLS = ["moh_code","moh_name","ds_name","ds_id","district","week_start","iso_week",
             "cases","incidence_per_100k","risk_tier","risk_tier_int",
             "centroid_lat","centroid_lon","birth_rate","area_km2"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f"Built {len(FEATURE_COLS)} features")


Built 63 features


## Train the 4 base models

Each model gets class weights (handles imbalance) and early stopping on the validation set.

###  LightGBM

In [6]:
train = df[df["week_start"] < "2024-01-01"].copy()
val   = df[(df["week_start"] >= "2024-01-01") & (df["week_start"] < "2025-01-01")].copy()
test  = df[df["week_start"] >= "2025-01-01"].copy()
print(f"Re-split: train={train.shape}, val={val.shape}, test={test.shape}")

Re-split: train=(118198, 78), val=(11752, 78), test=(16035, 78)


In [7]:
import lightgbm as lgb
X_train = train[FEATURE_COLS].fillna(0); y_train = train["risk_tier_int"].values
X_val   = val[FEATURE_COLS].fillna(0);   y_val   = val["risk_tier_int"].values
X_test  = test[FEATURE_COLS].fillna(0);  y_test  = test["risk_tier_int"].values
sw = compute_sample_weight("balanced", y_train)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

lgb_params = {
    "objective":"multiclass","num_class":4,"metric":"multi_logloss",
    "learning_rate":0.05,"num_leaves":63,"min_child_samples":30,
    "feature_fraction":0.8,"bagging_fraction":0.8,"bagging_freq":5,
    "reg_alpha":0.1,"reg_lambda":0.5,"n_jobs":-1,"verbose":-1,"seed":RANDOM_SEED,
}
print("Training LightGBM...")
dtrain = lgb.Dataset(X_train, y_train, weight=sw, categorical_feature=["district_cat"])
dval   = lgb.Dataset(X_val, y_val, reference=dtrain, categorical_feature=["district_cat"])
lgb_model = lgb.train(lgb_params, dtrain, num_boost_round=2000, valid_sets=[dval], valid_names=["val"],
                      callbacks=[lgb.early_stopping(80), lgb.log_evaluation(0)])
lgb_pred = lgb_model.predict(X_test).argmax(axis=1)
lgb_acc = accuracy_score(y_test, lgb_pred)
print(f"LightGBM test acc: {lgb_acc:.4f}  (best iter: {lgb_model.best_iteration})")


Train: (118198, 63)  Val: (11752, 63)  Test: (16035, 63)
Training LightGBM...
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[210]	val's multi_logloss: 0.604882
LightGBM test acc: 0.7573  (best iter: 210)


###  XGBoost

In [8]:
import xgboost as xgb
print("Training XGBoost...")
xgb_clf = xgb.XGBClassifier(
    n_estimators=2000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5, min_child_weight=5,
    objective="multi:softprob", num_class=4, random_state=RANDOM_SEED, n_jobs=-1,
    eval_metric="mlogloss", early_stopping_rounds=80, tree_method="hist",
)
xgb_clf.fit(X_train, y_train, sample_weight=sw, eval_set=[(X_val, y_val)], verbose=0)
xgb_pred = xgb_clf.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"XGBoost test acc: {xgb_acc:.4f}  (best iter: {xgb_clf.best_iteration})")


Training XGBoost...
XGBoost test acc: 0.7533  (best iter: 683)


###  CatBoost

In [9]:
from catboost import CatBoostClassifier
print("Training CatBoost (this takes a few minutes)...")
cat_clf = CatBoostClassifier(
    iterations=2000, depth=7, learning_rate=0.05,
    loss_function="MultiClass", eval_metric="MultiClass",
    random_seed=RANDOM_SEED, verbose=0, early_stopping_rounds=80, l2_leaf_reg=3.0,
    cat_features=["district_cat"], task_type="CPU",
)
cat_clf.fit(X_train, y_train, sample_weight=sw, eval_set=(X_val, y_val), use_best_model=True)
cat_pred = cat_clf.predict(X_test).astype(int).ravel()
cat_acc = accuracy_score(y_test, cat_pred)
print(f"CatBoost test acc: {cat_acc:.4f}")


Training CatBoost (this takes a few minutes)...
CatBoost test acc: 0.7608


###  LSTM (sequence model)

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEQ_LEN = 12
SEQ_FEATURES = ["cases","incidence_per_100k","district_total_lag1",
                "temp_avg","rain_1w","humidity","log_pop"]

# Build sequences
def build_sequences(frame, moh_list):
    X, y, d, m = [], [], [], []
    feat = frame[SEQ_FEATURES].values.astype(np.float32)
    for moh in moh_list:
        idx = sorted(frame.index[frame["moh_name"]==moh].tolist(),
                     key=lambda i: frame.at[i,"week_start"])
        vals = feat[idx]; tiers = frame.iloc[idx]["risk_tier_int"].values
        ds = frame.iloc[idx]["week_start"].values
        for i in range(SEQ_LEN, len(idx)):
            X.append(vals[i-SEQ_LEN:i]); y.append(tiers[i]); d.append(ds[i]); m.append(moh)
    return np.array(X), np.array(y), np.array(d), np.array(m)

print("Building LSTM sequences (12-week windows)...")
X_seq, y_seq, d_seq, m_seq = build_sequences(df, df["moh_name"].unique())
tr_m = d_seq < np.datetime64("2024-01-01")
va_m = (d_seq >= np.datetime64("2024-01-01")) & (d_seq < np.datetime64("2025-01-01"))
te_m = d_seq >= np.datetime64("2025-01-01")
Xtr, ytr = X_seq[tr_m], y_seq[tr_m]
Xva, yva = X_seq[va_m], y_seq[va_m]
Xte, yte = X_seq[te_m], y_seq[te_m]
mu = Xtr.reshape(-1, len(SEQ_FEATURES)).mean(axis=0)
sd = Xtr.reshape(-1, len(SEQ_FEATURES)).std(axis=0) + 1e-6
Xtr = (Xtr-mu)/sd; Xva = (Xva-mu)/sd; Xte = (Xte-mu)/sd

tf.random.set_seed(RANDOM_SEED)
lstm_model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, len(SEQ_FEATURES))),
    layers.LSTM(64, return_sequences=True), layers.Dropout(0.3),
    layers.LSTM(32), layers.Dropout(0.3),
    layers.Dense(32, activation="relu"), layers.Dropout(0.2),
    layers.Dense(4, activation="softmax"),
])
lstm_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cw = compute_class_weight("balanced", classes=np.arange(4), y=ytr)
print("Training LSTM...")
lstm_model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=256,
               class_weight=dict(enumerate(cw)),
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)],
               verbose=0)
lstm_probs = lstm_model.predict(Xte, verbose=0)
lstm_pred = lstm_probs.argmax(axis=1)
lstm_acc = accuracy_score(yte, lstm_pred)
print(f"LSTM test acc: {lstm_acc:.4f}  (test set is LSTM-eligible subset: {len(yte)} rows)")


Building LSTM sequences (12-week windows)...
Training LSTM...
LSTM test acc: 0.7345  (test set is LSTM-eligible subset: 16035 rows)


##  Stacking ensemble

A logistic regression meta learner combines the 4 base models. The 3 model version (no LSTM) is what we ship to production for speed - it's 0.2% less accurate but doesn't need sequence data at inference time.

In [11]:
# 3-model stack: LightGBM + XGBoost + CatBoost (12 features) - this is the PRODUCTION model
# 4-model stack: + LSTM (16 features) - for evaluation only, since LSTM is slower at inference

val_p3  = np.hstack([lgb_model.predict(X_val),  xgb_clf.predict_proba(X_val),  cat_clf.predict_proba(X_val)])
test_p3 = np.hstack([lgb_model.predict(X_test), xgb_clf.predict_proba(X_test), cat_clf.predict_proba(X_test)])

def make_aligned_df(frame, X, lgb_m, xgb_m, cat_m, lstm_pred=None, m_arr=None, d_arr=None, mask=None):
    keys = frame[["moh_name", "week_start"]].reset_index(drop=True)
    out = keys.copy()
    lgb_p = lgb_m.predict(X)
    for i in range(4): out[f"p{i}"] = lgb_p[:, i]
    xgb_p = xgb_m.predict_proba(X)
    for i in range(4): out[f"px{i}"] = xgb_p[:, i]
    cat_p = cat_m.predict_proba(X)
    for i in range(4): out[f"py{i}"] = cat_p[:, i]
    if lstm_pred is not None and m_arr is not None and d_arr is not None and mask is not None:
        lstm_df = pd.DataFrame({"moh_name": m_arr[mask], "week_start": d_arr[mask]})
        for i in range(4): lstm_df[f"pz{i}"] = lstm_pred[:, i]
        out = out.merge(lstm_df, on=["moh_name", "week_start"], how="inner")
    return out.merge(frame.reset_index(drop=True)[["moh_name", "week_start", "risk_tier_int"]],
                     on=["moh_name", "week_start"], how="inner")

val_aligned  = make_aligned_df(val,  X_val,  lgb_model, xgb_clf, cat_clf, lstm_model.predict(Xva, verbose=0), m_seq, d_seq, va_m)
test_aligned = make_aligned_df(test, X_test, lgb_model, xgb_clf, cat_clf, lstm_probs,                       m_seq, d_seq, te_m)

def to_4stack(df):
    return np.hstack([
        df[[f"p{i}"  for i in range(4)]].values,
        df[[f"px{i}" for i in range(4)]].values,
        df[[f"py{i}" for i in range(4)]].values,
        df[[f"pz{i}" for i in range(4)]].values,
    ])

val_p4  = to_4stack(val_aligned)
test_p4 = to_4stack(test_aligned)
y_val_4  = val_aligned["risk_tier_int"].values
y_test_4 = test_aligned["risk_tier_int"].values
print(f"3-model: val {val_p3.shape}, test {test_p3.shape}")
print(f"4-model: val {val_p4.shape}, test {test_p4.shape}  (LSTM-eligible only)")


meta_3 = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1, random_state=RANDOM_SEED,
                            class_weight="balanced", solver="lbfgs")
meta_3.fit(val_p3, y_val)
ens_3 = meta_3.predict(test_p3)
ens_3_acc = accuracy_score(y_test, ens_3)
print(f"\n3-model ensemble (PRODUCTION): {ens_3_acc:.4f}  on {len(y_test)} test rows")

meta_4 = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1, random_state=RANDOM_SEED,
                            class_weight="balanced", solver="lbfgs")
meta_4.fit(val_p4, y_val_4)
ens_4 = meta_4.predict(test_p4)
ens_4_acc = accuracy_score(y_test_4, ens_4)
print(f"4-model ensemble (eval only):  {ens_4_acc:.4f}  on {len(y_test_4)} LSTM-eligible rows")
print(f"3-model is the one we ship to production.")

3-model: val (11752, 12), test (16035, 12)
4-model: val (11752, 16), test (16035, 16)  (LSTM-eligible only)

3-model ensemble (PRODUCTION): 0.7676  on 16035 test rows
4-model ensemble (eval only):  0.7695  on 16035 LSTM-eligible rows
3-model is the one we ship to production.


## Performance summary

In [12]:
results = pd.DataFrame([
    ("Stacking Ensemble (3-model, prod)", ens_3_acc),
    ("CatBoost",          cat_acc),
    ("LightGBM",          lgb_acc),
    ("XGBoost",           xgb_acc),
    ("LSTM",              lstm_acc),
    ("Naive persistence (baseline)",  accuracy_score(y_test, df.groupby("moh_name")["risk_tier_int"].shift(1).loc[test.index].fillna(0).astype(int).values)),
], columns=["model","accuracy"]).sort_values("accuracy", ascending=False)

print(results.to_string(index=False))

print("\nProduction ensemble (3-model) per-class breakdown:")
print(classification_report(y_test, ens_3, target_names=TIER_ORDER, digits=4))

                            model  accuracy
Stacking Ensemble (3-model, prod)  0.767633
                         CatBoost  0.760836
                         LightGBM  0.757281
                          XGBoost  0.753290
     Naive persistence (baseline)  0.752479
                             LSTM  0.734456

Production ensemble (3-model) per-class breakdown:
              precision    recall  f1-score   support

         Low     0.8122    0.7687    0.7899      5327
       Watch     0.7698    0.7513    0.7605      6877
     Warning     0.7366    0.7978    0.7660      3393
       Alert     0.5611    0.7763    0.6513       438

    accuracy                         0.7676     16035
   macro avg     0.7199    0.7735    0.7419     16035
weighted avg     0.7712    0.7676    0.7684     16035



## Case count prediction (regression, alongside the risk tier)

Same features, same train/val/test split. Target is the raw weekly `cases` count. Trained in log1p space since case counts are right-skewed (lots of low-count weeks, a few large outbreaks), then converted back with expm1 at prediction time.

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_train_c = train["cases"].values.astype(float)
y_val_c   = val["cases"].values.astype(float)
y_test_c  = test["cases"].values.astype(float)

y_train_log = np.log1p(y_train_c)
y_val_log   = np.log1p(y_val_c)

print(f"Case counts - train: min {y_train_c.min():.0f}, max {y_train_c.max():.0f}, mean {y_train_c.mean():.1f}, median {np.median(y_train_c):.0f}")


Case counts - train: min 0, max 672, mean 4.8, median 1


###  LightGBM regressor

In [14]:
print("Training LightGBM regressor (case counts)...")
lgb_reg_params = {
    "objective": "regression", "metric": "mae",
    "learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 30,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 5,
    "reg_alpha": 0.1, "reg_lambda": 0.5, "n_jobs": -1, "verbose": -1, "seed": RANDOM_SEED,
}
dtrain_r = lgb.Dataset(X_train, y_train_log, categorical_feature=["district_cat"])
dval_r   = lgb.Dataset(X_val, y_val_log, reference=dtrain_r, categorical_feature=["district_cat"])
lgb_reg = lgb.train(lgb_reg_params, dtrain_r, num_boost_round=2000, valid_sets=[dval_r], valid_names=["val"],
                     callbacks=[lgb.early_stopping(80), lgb.log_evaluation(0)])
lgb_reg_pred = np.expm1(lgb_reg.predict(X_test)).clip(0)
lgb_reg_mae = mean_absolute_error(y_test_c, lgb_reg_pred)
print(f"LightGBM regressor test MAE: {lgb_reg_mae:.2f} cases  (best iter: {lgb_reg.best_iteration})")


Training LightGBM regressor (case counts)...
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[79]	val's l1: 0.236493
LightGBM regressor test MAE: 1.28 cases  (best iter: 79)


###  XGBoost regressor

In [15]:
print("Training XGBoost regressor (case counts)...")
xgb_reg = xgb.XGBRegressor(
    n_estimators=2000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5, min_child_weight=5,
    objective="reg:squarederror", random_state=RANDOM_SEED, n_jobs=-1,
    eval_metric="mae", early_stopping_rounds=80, tree_method="hist",
)
xgb_reg.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], verbose=0)
xgb_reg_pred = np.expm1(xgb_reg.predict(X_test)).clip(0)
xgb_reg_mae = mean_absolute_error(y_test_c, xgb_reg_pred)
print(f"XGBoost regressor test MAE: {xgb_reg_mae:.2f} cases  (best iter: {xgb_reg.best_iteration})")


Training XGBoost regressor (case counts)...
XGBoost regressor test MAE: 1.26 cases  (best iter: 85)


###  CatBoost regressor

In [16]:
from catboost import CatBoostRegressor
print("Training CatBoost regressor (case counts)...")
cat_reg = CatBoostRegressor(
    iterations=2000, depth=7, learning_rate=0.05,
    loss_function="MAE", eval_metric="MAE",
    random_seed=RANDOM_SEED, verbose=0, early_stopping_rounds=80, l2_leaf_reg=3.0,
    cat_features=["district_cat"], task_type="CPU",
)
cat_reg.fit(X_train, y_train_log, eval_set=(X_val, y_val_log), use_best_model=True)
cat_reg_pred = np.expm1(cat_reg.predict(X_test)).clip(0)
cat_reg_mae = mean_absolute_error(y_test_c, cat_reg_pred)
print(f"CatBoost regressor test MAE: {cat_reg_mae:.2f} cases")


Training CatBoost regressor (case counts)...
CatBoost regressor test MAE: 1.23 cases


##  Stacking ensemble (case counts)

In [17]:
from sklearn.linear_model import Ridge

# same 3-model stack as the tier classifier: LightGBM + XGBoost + CatBoost, meta-learner fit on val, scored on test
val_r3  = np.column_stack([lgb_reg.predict(X_val),  xgb_reg.predict(X_val),  cat_reg.predict(X_val)])
test_r3 = np.column_stack([lgb_reg.predict(X_test), xgb_reg.predict(X_test), cat_reg.predict(X_test)])

meta_reg = Ridge(alpha=1.0, random_state=RANDOM_SEED)
meta_reg.fit(val_r3, y_val_log)

ens_reg_pred = np.expm1(meta_reg.predict(test_r3)).clip(0)
ens_reg_mae  = mean_absolute_error(y_test_c, ens_reg_pred)
ens_reg_rmse = np.sqrt(mean_squared_error(y_test_c, ens_reg_pred))
_mask = y_test_c > 0
ens_reg_mape = np.mean(np.abs((y_test_c[_mask] - ens_reg_pred[_mask]) / y_test_c[_mask])) * 100

print(f"\nCase count ensemble (PRODUCTION): MAE {ens_reg_mae:.2f}  RMSE {ens_reg_rmse:.2f}  MAPE {ens_reg_mape:.1f}%  on {len(y_test_c)} test rows")



Case count ensemble (PRODUCTION): MAE 1.23  RMSE 2.98  MAPE 33.6%  on 16035 test rows


## Performance summary (case counts)

In [18]:
naive_reg_pred = df.groupby("moh_name")["cases"].shift(1).loc[test.index].fillna(0).values

reg_results = pd.DataFrame([
    ("Stacking Ensemble (3-model, prod)", ens_reg_mae),
    ("CatBoost",  mean_absolute_error(y_test_c, cat_reg_pred)),
    ("LightGBM",  mean_absolute_error(y_test_c, lgb_reg_pred)),
    ("XGBoost",   mean_absolute_error(y_test_c, xgb_reg_pred)),
    ("Naive persistence (baseline)", mean_absolute_error(y_test_c, naive_reg_pred)),
], columns=["model","test_mae"]).sort_values("test_mae")

print(reg_results.to_string(index=False))


                            model  test_mae
                         CatBoost  1.225620
Stacking Ensemble (3-model, prod)  1.232127
     Naive persistence (baseline)  1.245837
                          XGBoost  1.261683
                         LightGBM  1.282041


## Save the deployment model


In [19]:
print("Saving deployment artifacts...")

lgb_model.save_model(f"{MODEL_DIR}/lgb_classifier.txt")
xgb_clf.save_model(f"{MODEL_DIR}/xgb_classifier.json")
cat_clf.save_model(f"{MODEL_DIR}/cat_classifier.cbm")
lstm_model.save(f"{MODEL_DIR}/lstm_model.keras")

lgb_reg.save_model(f"{MODEL_DIR}/lgb_regressor.txt")
xgb_reg.save_model(f"{MODEL_DIR}/xgb_regressor.json")
cat_reg.save_model(f"{MODEL_DIR}/cat_regressor.cbm")

with open(f"{MODEL_DIR}/meta_3model.pkl", "wb") as f: pickle.dump(meta_3, f)
with open(f"{MODEL_DIR}/meta_4model.pkl", "wb") as f: pickle.dump(meta_4, f)
with open(f"{MODEL_DIR}/meta_reg.pkl", "wb") as f: pickle.dump(meta_reg, f)

lstm_stats = {"mean": mu.tolist(), "std": sd.tolist(), "seq_features": SEQ_FEATURES, "seq_len": SEQ_LEN}
with open(f"{MODEL_DIR}/lstm_norm_stats.pkl", "wb") as f: pickle.dump(lstm_stats, f)

meta_json = {
    "feature_cols": FEATURE_COLS,
    "categorical_cols": ["district_cat"],
    "tier_to_int": TIER_TO_INT,
    "int_to_tier": {str(k): v for k, v in INT_TO_TIER.items()},
    "tier_thresholds": [float(T1), float(T2), float(T3)],
    "alert_decision_threshold": 0.5,
    "seq_features": SEQ_FEATURES,
    "seq_len": SEQ_LEN,
    "random_seed": RANDOM_SEED,
    "district_to_idx": district_to_idx,
    "case_count_log_transform": True,
}
with open(f"{MODEL_DIR}/pipeline_meta.json", "w") as f: json.dump(meta_json, f, indent=2)

print(f"\nSaved to {MODEL_DIR}/:")
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(f"{MODEL_DIR}/{f}") / 1024
    print(f"  {f:30s}  {size:>8.1f} KB")

if IN_COLAB:
    print("\nTo download all artifacts in Colab:")
    print("  from google.colab import files")
    print("  !zip -r models.zip models/")
    print("  files.download('models.zip')")


Saving deployment artifacts...

Saved to models/:
  cat_classifier.cbm                4481.1 KB
  cat_regressor.cbm                  363.6 KB
  lgb_classifier.txt                5640.2 KB
  lgb_regressor.txt                  454.9 KB
  lstm_model.keras                   418.6 KB
  lstm_norm_stats.pkl                  0.3 KB
  meta_3model.pkl                      1.1 KB
  meta_4model.pkl                      1.3 KB
  meta_reg.pkl                         0.5 KB
  pipeline_meta.json                   2.3 KB
  xgb_classifier.json              16872.4 KB
  xgb_regressor.json                1102.8 KB

To download all artifacts in Colab:
  from google.colab import files
  !zip -r models.zip models/
  files.download('models.zip')


## How to use this in production

The saved model expects a **specific input format** and returns a **specific output format**.

### What the model expects as input

For each (MOH area, current week) pair, you need a row with these 63 columns (in any order - the model uses names not positions):

| Group | Features | What they are |
|---|---|---|
| Case lags | `cases_lag1`, `cases_lag2`, `cases_lag3`, `cases_lag4`, `cases_lag5`, `cases_lag8`, `cases_lag12`, `cases_lag26`, `cases_lag52` | Cases from N weeks ago. `lag1` = last week, `lag52` = same week last year |
| Case rolling stats | `cases_roll{4,8,12}_{mean,max,std}` | Rolling mean/max/std over last N weeks (excludes current week) |
| Growth | `case_growth_wow`, `case_accel`, `case_trend_8w` | Week-over-week growth, its change, and 8-week linear trend |
| Incidence | `inc_lag1`, `inc_lag2`, `inc_lag4`, `inc_lag8`, `inc_roll4_mean`, `inc_roll12_mean` | Same as case lags but expressed as cases per 100k people |
| Seasonality | `iso_year`, `month`, `month_sin`, `month_cos`, `woy`, `woy_sin`, `woy_cos` | Year, month (1-12), week-of-year (1-52) as both raw and sin/cos encoded |
| District | `district_total_lag1/2/4`, `district_mean_lag1`, `district_max_lag1`, `district_total_roll4/12` | Same-week-but-LAST-WEEK stats for your district. `lag1` = last week, `lag4` = 4 weeks back |
| Spatial rank | `district_rank_lag1`, `district_zscore_lag1` | Your MOH's percentile rank and z-score within its district (using last week's data) |
| Weather | `temp_avg`, `temp_max`, `temp_min`, `temp_range`, `temp_avg_4w`, `humidity`, `humidity_4w`, `rain_1w`, `rain_2w`, `rain_4w`, `rain_change`, `heat_index`, `rain_x_temp`, `rain_x_humid` | Current weather + 4-week averages + interactions |
| Population | `population`, `pop_density`, `log_pop`, `log_density` | MOH-level population stats |
| Other | `weeks_since_outbreak_lag1` | Weeks since cases > 20 in this MOH (capped at 52) |
| Categorical | `district_cat` | District encoded as integer (0-24) |

**Required from the raw CSV (these are not features, but you need them to compute features):** `moh_name`, `district`, `week_start`, `cases`, `incidence_per_100k`, `population`, `temp_avg`, `temp_max`, `temp_min`, `humidity`, `rain_1w`, `rain_2w`, `rain_4w`, `temp_avg_4w`, `humidity_4w`

**The model predicts:** the tier (Low/Watch/Warning/Alert) **and** the expected case count for the current week, given features computed from prior weeks.

### Inference function


In [20]:
class DengueRadarPredictor:
    def __init__(self, model_dir="models"):
        import lightgbm as lgb, xgboost as xgb
        from catboost import CatBoostClassifier, CatBoostRegressor
        self.lgb = lgb.Booster(model_file=f"{model_dir}/lgb_classifier.txt")
        self.xgb = xgb.XGBClassifier(); self.xgb.load_model(f"{model_dir}/xgb_classifier.json")
        self.cat = CatBoostClassifier(); self.cat.load_model(f"{model_dir}/cat_classifier.cbm")
        with open(f"{model_dir}/meta_3model.pkl","rb") as f: self.meta = pickle.load(f)

        self.lgb_reg = lgb.Booster(model_file=f"{model_dir}/lgb_regressor.txt")
        self.xgb_reg = xgb.XGBRegressor(); self.xgb_reg.load_model(f"{model_dir}/xgb_regressor.json")
        self.cat_reg = CatBoostRegressor(); self.cat_reg.load_model(f"{model_dir}/cat_regressor.cbm")
        with open(f"{model_dir}/meta_reg.pkl","rb") as f: self.meta_reg = pickle.load(f)

        with open(f"{model_dir}/pipeline_meta.json") as f: self.meta_json = json.load(f)
        self.feature_cols = self.meta_json["feature_cols"]
        self.int_to_tier = {int(k): v for k, v in self.meta_json["int_to_tier"].items()}
        self.alert_threshold = self.meta_json.get("alert_decision_threshold", 0.5)
        print(f"Loaded predictor from '{model_dir}' ({len(self.feature_cols)} features)")

    def predict(self, df):
        X = df[self.feature_cols].fillna(0)
        p = np.hstack([self.lgb.predict(X), self.xgb.predict_proba(X), self.cat.predict_proba(X)])
        probs = self.meta.predict_proba(p)
        return probs.argmax(axis=1), probs

    def predict_with_threshold(self, df):

        pred, probs = self.predict(df)
        pred = pred.copy()
        pred[(pred == 3) & (probs[:, 3] < self.alert_threshold)] = 2
        return pred, probs

    def predict_cases(self, df):
        X = df[self.feature_cols].fillna(0)
        p = np.column_stack([self.lgb_reg.predict(X), self.xgb_reg.predict(X), self.cat_reg.predict(X)])
        pred_log = self.meta_reg.predict(p)
        return np.expm1(pred_log).clip(0)

    def predict_week(self, week_start, frame):
        rows = frame[frame["week_start"] == pd.Timestamp(week_start)]
        if rows.empty: return pd.DataFrame()
        pred, probs = self.predict(rows)
        cases = self.predict_cases(rows)
        out = rows[["moh_name","district"]].copy().reset_index(drop=True)
        out["predicted_tier"] = [self.int_to_tier[p] for p in pred]
        out["predicted_cases"] = np.round(cases).astype(int)
        out[["p_Low","p_Watch","p_Warning","p_Alert"]] = probs
        out["alert_high_confidence"] = out["p_Alert"] > self.alert_threshold
        return out.sort_values("p_Alert", ascending=False)

predictor_code = '''
import json, pickle, numpy as np, pandas as pd
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor

class DengueRadarPredictor:
    def __init__(self, model_dir="models"):
        self.lgb = lgb.Booster(model_file=f"{model_dir}/lgb_classifier.txt")
        self.xgb = xgb.XGBClassifier(); self.xgb.load_model(f"{model_dir}/xgb_classifier.json")
        self.cat = CatBoostClassifier(); self.cat.load_model(f"{model_dir}/cat_classifier.cbm")
        with open(f"{model_dir}/meta_3model.pkl","rb") as f: self.meta = pickle.load(f)

        self.lgb_reg = lgb.Booster(model_file=f"{model_dir}/lgb_regressor.txt")
        self.xgb_reg = xgb.XGBRegressor(); self.xgb_reg.load_model(f"{model_dir}/xgb_regressor.json")
        self.cat_reg = CatBoostRegressor(); self.cat_reg.load_model(f"{model_dir}/cat_regressor.cbm")
        with open(f"{model_dir}/meta_reg.pkl","rb") as f: self.meta_reg = pickle.load(f)

        with open(f"{model_dir}/pipeline_meta.json") as f: self.meta_json = json.load(f)
        self.feature_cols = self.meta_json["feature_cols"]
        self.int_to_tier = {int(k): v for k, v in self.meta_json["int_to_tier"].items()}
        self.alert_threshold = self.meta_json.get("alert_decision_threshold", 0.5)

    def predict(self, df):
        X = df[self.feature_cols].fillna(0)
        p = np.hstack([self.lgb.predict(X), self.xgb.predict_proba(X), self.cat.predict_proba(X)])
        probs = self.meta.predict_proba(p)
        return probs.argmax(axis=1), probs

    def predict_with_threshold(self, df):
        pred, probs = self.predict(df)
        pred = pred.copy()
        pred[(pred == 3) & (probs[:, 3] < self.alert_threshold)] = 2
        return pred, probs

    def predict_cases(self, df):
        X = df[self.feature_cols].fillna(0)
        p = np.column_stack([self.lgb_reg.predict(X), self.xgb_reg.predict(X), self.cat_reg.predict(X)])
        pred_log = self.meta_reg.predict(p)
        return np.expm1(pred_log).clip(0)

    def predict_week(self, week_start, frame):
        rows = frame[frame["week_start"] == pd.Timestamp(week_start)]
        if rows.empty: return pd.DataFrame()
        pred, probs = self.predict(rows)
        cases = self.predict_cases(rows)
        out = rows[["moh_name","district"]].copy().reset_index(drop=True)
        out["predicted_tier"] = [self.int_to_tier[p] for p in pred]
        out["predicted_cases"] = np.round(cases).astype(int)
        out[["p_Low","p_Watch","p_Warning","p_Alert"]] = probs
        out["alert_high_confidence"] = out["p_Alert"] > self.alert_threshold
        return out.sort_values("p_Alert", ascending=False)
'''
with open("dengueradar_predictor.py", "w") as f:
    f.write(predictor_code)
print("Wrote dengueradar_predictor.py - drop this in your deployment repo.")


Wrote dengueradar_predictor.py - drop this in your deployment repo.


### Demo: predict for one MOH-week, then for a whole week

In [21]:
predictor = DengueRadarPredictor(MODEL_DIR)

example = test.iloc[[10]]
pred, probs = predictor.predict(example)
pred_cases = predictor.predict_cases(example)
print(f"\n=== Single prediction ===")
print(f"MOH: {example['moh_name'].iloc[0]}  Week: {example['week_start'].iloc[0].date()}")
print(f"True tier:      {INT_TO_TIER[example['risk_tier_int'].iloc[0]]}")
print(f"Predicted tier: {INT_TO_TIER[pred[0]]}")
print(f"True cases:      {example['cases'].iloc[0]:.0f}")
print(f"Predicted cases: {pred_cases[0]:.0f}")
print("Probabilities:")
for i, t in enumerate(TIER_ORDER):
    print(f"  {t:8s}: {probs[0, i]:.3f}")

print(f"\n=== Predict for week of 2025-06-07 ===")
weekly = predictor.predict_week("2025-06-07", df)
print(f"Predictions for {len(weekly)} MOHs")
print(f"Tier breakdown: {weekly['predicted_tier'].value_counts().to_dict()}")
print(f"Total predicted cases across all MOHs: {weekly['predicted_cases'].sum():.0f}")
print(f"\nTop 10 highest-Alert-confidence MOHs:")
print(weekly.head(10)[["moh_name","district","predicted_tier","predicted_cases","p_Alert"]].to_string(index=False))


Loaded predictor from 'models' (63 features)

=== Single prediction ===
MOH: Agalawatta  Week: 2025-03-15
True tier:      Low
Predicted tier: Low
True cases:      1
Predicted cases: 1
Probabilities:
  Low     : 0.647
  Watch   : 0.292
  Alert   : 0.009

=== Predict for week of 2025-06-07 ===
Predictions for 226 MOHs
Tier breakdown: {'Watch': 112, 'Warning': 75, 'Low': 32, 'Alert': 7}
Total predicted cases across all MOHs: 1262

Top 10 highest-Alert-confidence MOHs:
       moh_name    district predicted_tier  predicted_cases  p_Alert
         Eravur  Batticaloa          Alert               52 0.986996
Gomarankadawala Trincomalee          Alert                4 0.963662
    Trincomalee Trincomalee          Alert               13 0.963281
        Kinniya Trincomalee          Alert               16 0.959483
         Nallur      Jaffna          Alert                8 0.913743
      Mc jaffna      Jaffna          Alert               17 0.879948
      Poonakary Kilinochchi          Alert     



**Production deployment checklist:**

- [ ] Set up a weekly cron job that pulls latest weather + last 52 weeks of cases
- [ ] Compute the 63 features for all 226 MOHs
- [ ] Call `predictor.predict_week("next_monday", feature_df)`
- [ ] Filter to `alert_high_confidence == True` for the alert system
- [ ] Send to dashboard / WhatsApp / SMS
- [ ] Log predictions for accuracy monitoring
- [ ] Schedule a quarterly retraining job

**Files saved in `models/`:**
- `lgb_classifier.txt` - LightGBM (risk tier)
- `xgb_classifier.json` - XGBoost (risk tier)
- `cat_classifier.cbm` - CatBoost (risk tier)
- `lstm_model.keras` - LSTM (optional, used in 4-model eval)
- `meta_3model.pkl` - Stacking meta-learner, risk tier (3-model, **production**)
- `meta_4model.pkl` - 4-model version (for evaluation only)
- `lgb_regressor.txt` - LightGBM (case count)
- `xgb_regressor.json` - XGBoost (case count)
- `cat_regressor.cbm` - CatBoost (case count)
- `meta_reg.pkl` - Stacking meta-learner, case count (**production**)
- `pipeline_meta.json` - Feature names, encodings, paths (**read this first**)
- `lstm_norm_stats.pkl` - LSTM normalization constants

